In [ ]:

# --- Two Kaggle-specific traps, both already hit once. ---
# 1. Kaggle gives a Tesla P100 (compute capability 6.0, Pascal). Recent torch
#    wheels dropped sm_60, so a stock `pip install ultralytics` installs a torch
#    that cannot run on the GPU it was given: training dies at .to(device) with
#    "CUDA error: no kernel image is available for execution on the device".
#    Fix: pin a cu121 build BEFORE torch is first imported. Never os.execv to
#    apply it -- that kills the Jupyter kernel (DeadKernelError).
# 2. Ultralytics auto-registers a Ray Tune callback whenever `ray` is importable.
#    Kaggle ships a ray whose internal API no longer matches, so the callback
#    raises at the END OF EPOCH 1:
#      AttributeError: module 'ray.train._internal.session' has no attribute '_get_session'
#    Fix: remove the integration packages before ultralytics is imported.
import subprocess, sys
info = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap,memory.total",
                       "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", info or "NONE")
cap = info.split(",")[1].strip() if "," in info else ""
assert "torch" not in sys.modules, "torch already imported -- the pin would not take effect"

if cap.startswith("6."):
    print(f"compute capability {cap} is Pascal: pinning torch 2.5.1 + cu121 (has sm_60)")
    subprocess.run([sys.executable,"-m","pip","-q","install",
                    "torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"], check=True)
else:
    print(f"compute capability {cap}: stock torch is fine")

# Strip every optional integration ultralytics might auto-hook. Only ray has
# actually bitten us, but any of these can raise inside a training callback and
# lose hours of work at an epoch boundary.
subprocess.run([sys.executable,"-m","pip","-q","uninstall","-y",
                "ray","wandb","comet_ml","mlflow","dvclive","neptune","clearml"], check=False)
subprocess.run([sys.executable,"-m","pip","-q","install","ultralytics==8.3.40"], check=True)

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| arch list", torch.cuda.get_arch_list())
assert torch.cuda.is_available(), "no CUDA device"
_x = (torch.randn(64,64,device="cuda") @ torch.randn(64,64,device="cuda")).sum().item()
torch.cuda.synchronize()
print("CUDA smoke test PASSED:", round(_x,3))

# Prove the ray callback is gone before committing to an 8-hour run.
import ultralytics
from ultralytics.utils import callbacks as _cb
hooked = {c.__module__ for v in _cb.default_callbacks.values() for c in v}
print("ultralytics", ultralytics.__version__)
print("callback modules registered:", sorted(m.split(".")[-1] for m in hooked))
assert not any("raytune" in m for m in hooked), "ray callback still registered"
print("ray callback absent. safe to train.")


In [ ]:

import os, json, glob, time
ROOT = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if "data.yaml" in fns and dp.rstrip("/").endswith("1class"):
        ROOT = dp; break
assert ROOT, "dataset not found under /kaggle/input"
print("data root:", ROOT)
YAML = "/kaggle/working/data.yaml"
with open(YAML,"w") as fh:
    fh.write(f"path: {ROOT}\ntrain: train/images\nval: valid/images\ntest: test/images\n\n")
    fh.write("nc: 1\nnames: ['burn']\n")
for sp in ("train","valid","test"):
    print(sp, len(glob.glob(f"{ROOT}/{sp}/images/*")))

def sid(n): return n.split("_jpg.rf.")[0] if "_jpg.rf." in n else n.rsplit(".",1)[0]
S = {sp:{sid(os.path.basename(p)) for p in glob.glob(f"{ROOT}/{sp}/images/*")}
     for sp in ("train","valid","test")}
bad = (S["train"]&S["valid"])|(S["train"]&S["test"])|(S["valid"]&S["test"])
print("source overlap between folds:", len(bad))
assert not bad, "THIS DATASET STILL LEAKS -- refusing to train"


In [ ]:

from ultralytics import YOLO
# One epoch measured at 4:45 on a P100, so 100 epochs is ~8 h against Kaggle's
# 12 h ceiling. `time` is a hard wall-clock stop that still writes best.pt, so a
# slow run degrades into a shorter one instead of a dead kernel. If it fires we
# report the truncation; less training for the clean model is conservative with
# respect to this paper's conclusion.
t0 = time.time()
YOLO("yolov8x-seg.pt").train(
    data=YAML, epochs=100, time=10.0, imgsz=640, batch=8, optimizer="AdamW",
    lr0=1e-3, seed=42, patience=20, project="/kaggle/working/runs",
    name="seg1", exist_ok=True, verbose=True)
hrs = (time.time()-t0)/3600
print(f"train wall-clock: {hrs:.2f} h")

best = "/kaggle/working/runs/seg1/weights/best.pt"
m = YOLO(best)
out = {"variant":"1class","train_hours":round(hrs,3),
       "split":"source-grouped leak-free","hit_time_cap": hrs >= 9.9}
for split in ("val","test"):
    r = m.val(data=YAML, split=split, imgsz=640, verbose=False)
    out[f"{split}_mask_map50"] = float(r.seg.map50)
    out[f"{split}_mask_map"]   = float(r.seg.map)
    print(split, "mask mAP50", round(out[f"{split}_mask_map50"],4),
          "mAP50-95", round(out[f"{split}_mask_map"],4))
json.dump(out, open("/kaggle/working/seg_1class_results.json","w"), indent=1)
print(json.dumps(out, indent=1))
print("CONTAMINATED baseline: 1class test mask mAP50 0.726, 3class 0.603")


In [ ]:

import numpy as np, cv2
from ultralytics import YOLO
m = YOLO("/kaggle/working/runs/seg1/weights/best.pt")
os.makedirs("/kaggle/working/masked_test", exist_ok=True)
meta=[]
for p in sorted(glob.glob(f"{ROOT}/test/images/*")):
    img=cv2.imread(p); H,W=img.shape[:2]
    r=m.predict(p, conf=0.05, verbose=False)[0]
    if r.masks is None or len(r.masks.data)==0:
        out=img; found=False            # full-frame fallback: deployed behaviour
    else:
        mk=r.masks.data.cpu().numpy().max(0)
        mk=cv2.resize(mk,(W,H),interpolation=cv2.INTER_NEAREST)>0.5
        out=img*mk[...,None]; found=True
    cv2.imwrite(f"/kaggle/working/masked_test/{os.path.basename(p)}", out)
    meta.append({"img":os.path.basename(p),"detected":bool(found)})
json.dump(meta, open("/kaggle/working/masked_test_meta.json","w"), indent=1)
print("masked test images:", len(meta),
      "| no detection:", sum(1 for x in meta if not x["detected"]))
